# 补充结论：后半段中国盈余公告与正文未列表结果

在03之后运行。利润表公告自2007年开始，SUE还需要同比差分和八季方差，因此中国Table1这里只覆盖具备历史的后半段，不能当作2000—2016全样本。美国仍缺数据。

使用作者在线附录允许的扣非盈利缺失回退 `NPParentComp`（不含少数股东损益净利润）。把Q2/Q3/Q4按年内累计值与前季度差分拆季，Q1直接用本期值；累计释义仍需RESSET词典核验，明确标记推断。派生季度可用日是两份构成报表公告日的最大值。SUE与八季分母的所有构成报表必须在本次公告日可见。

来源：[作者在线附录](https://faculty.wharton.upenn.edu/wp-content/uploads/2016/11/size_value_china_appendix_2_rev.pdf)。

In [ ]:
income_columns = ['A_StkCd','EndDt','InfoPubDt','ReportType','ConFlg','AdjFlg','NPParentComp']
income = read_paper_statement('RESSET_IS_ALL_1.csv',income_columns)
income = income[income['ReportType'].isin(['Q1','Q2','Q3','Q4'])].sort_values(['A_StkCd','EndDt','InfoPubDt']).drop_duplicates(['A_StkCd','EndDt'],keep='first').copy()
income['previous_end'] = income['EndDt']-pd.offsets.QuarterEnd(1)
previous = income[['A_StkCd','EndDt','InfoPubDt','NPParentComp']].rename(columns={'EndDt':'previous_end','InfoPubDt':'previous_public','NPParentComp':'previous_earnings'})
income = income.merge(previous,on=['A_StkCd','previous_end'],how='left',validate='many_to_one')
same_year = income['previous_end'].dt.year.eq(income['EndDt'].dt.year)
income['earnings'] = income['NPParentComp']
income.loc[same_year,'earnings'] = income.loc[same_year,'NPParentComp']-income.loc[same_year,'previous_earnings']
income['earnings_available_date'] = income['InfoPubDt']
income.loc[same_year,'earnings_available_date'] = income.loc[same_year,['InfoPubDt','previous_public']].max(axis=1)
income['fiscal_quarter'] = income['EndDt'].dt.to_period('Q')
cn_earnings = income[['A_StkCd','fiscal_quarter','InfoPubDt','earnings','earnings_available_date']].rename(columns={'A_StkCd':stock_code,'InfoPubDt':'announcement_date'})
cn_daily = df[[stock_code,date,daily_ret]].rename(columns={daily_ret:'stock_return'})
cn_market = daily_market[['market_return']].reset_index()
cn_caps = snap[[stock_code,form,monthly_market_value]].rename(columns={form:'cap_month',monthly_market_value:'market_value'})
partial_earnings_response = get_earnings_response(cn_earnings,cn_daily,cn_market,cn_caps,'China',False)
extension_results['table1_cn'] = partial_earnings_response
for row in partial_earnings_response['table'].to_dict('records'):
    panel = 'k0_china' if row['panel']=='A' else 'k3_china'
    for column,metric in [('b_x100','beta'),('t','t_beta'),('r2','r2')]:
        record_result('table1',panel,row['size_group'],metric,row[column],row['months'],status='后半段归母盈余回退/拆季推断；实际覆盖见08')
extension_coverage_rows = [row for row in extension_coverage_rows if row['target']!='table1_cn']
extension_coverage_rows.append({'target':'table1_cn','status':'后半段样本已计算，非完整原表','detail':'NPParentComp回退/拆季；8季历史与公告可见性筛选；美国仍缺'})
extension_coverage = pd.DataFrame(extension_coverage_rows)
display(partial_earnings_response['table'])


## 壳市值占比、负EP与剔除负EP的因子稳健性

原文约7%与15%分别是市值占比和股票数占比，不能混算。负EP占比同时给整个股票池和有效EP两种分母。负EP股票对正EP构造VMG的回归分别报告仅价值与三因子控制；原文未完整列出该规格，不能混成精确数值复现。

剔负EP稳健性这里计算因子溢价和FF-3互解释，不自动承认“原文所有异象均稳健”。

In [ ]:
shell_cap_share = monthly_all.groupby(ym).apply(lambda group: group.loc[group['size_percentile'].le(0.30),monthly_market_value].sum()/group[monthly_market_value].sum())
negative_ep_counts = monthly_final.groupby(ym)[monthly_ep].agg(total='size',valid='count',negative=lambda x:x.lt(0).sum())
negative_ep_counts['fraction_total'] = negative_ep_counts['negative']/negative_ep_counts['total']
negative_ep_counts['fraction_valid'] = negative_ep_counts['negative']/negative_ep_counts['valid']
print('最小30%市值占比（月均）：',shell_cap_share.mean(),'；原文约0.07')
print('负EP比例（月均）：',negative_ep_counts[['fraction_total','fraction_valid']].mean().to_dict(),'；原文约0.15')
positive_ep_panel = monthly_final[monthly_final[monthly_ep].gt(0)].copy()
positive_ports,positive_size,positive_value = get_size_value_ports(positive_ep_panel,monthly_ep)
positive_market = positive_ep_panel.groupby(ym)[[monthly_market_value,monthly_excess_ret]].apply(get_vw_return)
f_positive_ep = pd.DataFrame({mkt:positive_market,smb:positive_size,vmg:positive_value})
negative_ep_return = monthly_final[monthly_final[monthly_ep].lt(0)].groupby(ym)[[monthly_market_value,monthly_excess_ret]].apply(get_vw_return)
negative_loading_rows = []
for name,factors in [('value_only',f_positive_ep[[vmg]]),('positive_ep_ch3',f_positive_ep)]:
    result = run_ts(negative_ep_return,factors)
    negative_loading_rows.append({'specification':name,'beta_vmg':result.params[vmg],'t_beta_vmg':result.tvalues[vmg],'months':int(result.nobs),'paper_beta':-0.28,'paper_t':-3.31})
display(pd.DataFrame(negative_loading_rows))
display(pd.DataFrame({'mean_pct':100*f_positive_ep.mean(),'std_pct':100*f_positive_ep.std(),'mean_t':f_positive_ep.mean()/(f_positive_ep.std()/np.sqrt(f_positive_ep.count())),'months':f_positive_ep.count()}))
positive_bm_ports,positive_ff_size,positive_ff_value = get_size_value_ports(positive_ep_panel,monthly_bm)
f_positive_ff3 = pd.DataFrame({mkt:positive_market,'ff_size_factor':positive_ff_size,'ff_value_factor':positive_ff_value})
for column in [smb,vmg]:
    result = run_ts(f_positive_ep[column],f_positive_ff3)
    print('剔负EP {} 对 FF-3：alpha {:.4f}% / t {:.3f} / {}月'.format(column,100*result.params['const'],result.tvalues['const'],int(result.nobs)))


## 异象十分位解释度与动量

CH-3/市场分别解释异象组合约90%/85%方差是正文未列表结论。这里对每个组合用相同月份回归、再对组合平均。反转只保存了多空，未保存十条单独持仓腿，所以明确只覆盖其余九个正文异象的90个组合。

动量是过去第2—12个自然月复利信号，赢家减输家。历史恢复完整月历；缺月不压缩窗口，预热不足的早期月份保持缺失。

In [ ]:
variance_rows = []
for name in [name for name in main_anomalies if name!='reversal']:
    for decile in range(1,11):
        column = '{}_{}'.format(name,decile)
        if column not in portfolios_uncon:
            continue
        sample = pd.concat([portfolios_uncon[column].rename('test_return'),f_ch3],axis=1).dropna()
        if len(sample)<5:
            continue
        for model,factors in [('ch3',sample[[mkt,smb,vmg]]),('capm',sample[[mkt]])]:
            result = run_ts(sample['test_return'],factors)
            variance_rows.append({'anomaly':name,'decile':decile,'model':model,'r2':result.rsquared,'months':int(result.nobs)})
portfolio_variance = pd.DataFrame(variance_rows)
display(portfolio_variance.groupby('model').agg(mean_r2=('r2','mean'),portfolios=('r2','size'),min_months=('months','min'),max_months=('months','max')))
monthly_history = monthly.pivot(index=ym,columns=stock_code,values=monthly_return).reindex(pd.period_range('2000-01','2016-12',freq='M'))
momentum_signal = (1+monthly_history).rolling(11,min_periods=11).apply(np.prod,raw=True).shift(2)-1
momentum_signal = momentum_signal.rename_axis(index=ym,columns=stock_code).stack().rename('momentum_signal').reset_index()
momentum_panel = monthly_final.merge(momentum_signal,on=[stock_code,ym],how='left',validate='one_to_one').dropna(subset=['momentum_signal'])
momentum_panel['momentum_group'] = momentum_panel.groupby(ym)['momentum_signal'].transform(lambda x:get_quantile_labels(x,10))
momentum_ports = momentum_panel.dropna(subset=['momentum_group']).groupby([ym,'momentum_group'])[[monthly_market_value,monthly_excess_ret]].apply(get_vw_return).unstack().reindex(columns=range(1,11))
momentum_return = momentum_ports[10]-momentum_ports[1]
for name,factors in [('capm',capm),('ch3',f_ch3)]:
    result = run_ts(momentum_return,factors)
    print('动量 {}：月alpha {:.4f}% / t {:.3f} / {}月'.format(name,100*result.params['const'],result.tvalues['const'],int(result.nobs)))
